# Grocery Intelligence — Demo Notebook

This notebook demonstrates the full search and recommendation pipeline:

1. **Data Overview** — 49,688 real Instacart products enriched with Open Food Facts nutrition data
2. **Hybrid Search** — BM25 + Semantic embeddings + Reciprocal Rank Fusion
3. **XGBoost Learn-to-Rank** — Feature-based reranking with 12 signals
4. **Substitute Recommendations** — Find similar, healthier, or popular alternatives
5. **Evaluation** — Precision@5 comparison across all methods

In [ ]:
import sys, os, warnings
sys.path.insert(0, os.path.abspath(".."))
warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import numpy as np
import pandas as pd

# Load product catalog and embeddings
catalog = pd.read_parquet("../data/processed/product_catalog.parquet")
embeddings = np.load("../data/embeddings/product_embeddings.npy")

print(f"Products: {len(catalog):,}")
print(f"Embeddings: {embeddings.shape}")
print(f"Columns: {list(catalog.columns)}")
catalog.head(3)

## 1. Data Overview

The catalog contains 49,688 real products from Instacart, enriched with nutrition data from Open Food Facts.

In [ ]:
print(f"Total products:       {len(catalog):,}")
print(f"Categories:           {catalog['category'].nunique()}")
print(f"Departments:          {catalog['department'].nunique()}")
print(f"With nutrition data:  {catalog['calories_100g'].notna().sum():,} ({catalog['calories_100g'].notna().mean():.1%})")
print(f"With ingredients:     {catalog['ingredients'].notna().sum():,} ({catalog['ingredients'].notna().mean():.1%})")
print(f"\nTop 10 categories by product count:")
catalog["category"].value_counts().head(10)

## 2. Hybrid Search Demo

The search engine combines **BM25 keyword matching** with **semantic embeddings** using **Reciprocal Rank Fusion (RRF)**.

This means queries like "healthy breakfast" find relevant products even when the exact words don't appear in product names.

In [ ]:
from src.models.embeddings import ProductEmbedder
from src.search.engine import GrocerySearchEngine

embedder = ProductEmbedder()
engine = GrocerySearchEngine(catalog=catalog, embeddings=embeddings, embedder=embedder)

# Demo: search for "low sugar yogurt"
results = engine.search("low sugar yogurt", top_k=5, use_reranker=False)
print("Query: 'low sugar yogurt'\n")
for i, r in enumerate(results, 1):
    sugar = f"{r['sugar_100g']:.1f}g sugar" if pd.notna(r.get("sugar_100g")) else "no data"
    print(f"  {i}. {r['product_name']:<45} [{r['category']}] ({sugar})")

In [ ]:
# Semantic understanding — these queries don't use exact product name keywords
demo_queries = [
    "Korean BBQ ingredients",
    "healthy snack for kids",
    "something to drink that's not soda",
]

for query in demo_queries:
    results = engine.search(query, top_k=3, use_reranker=False)
    print(f"Query: '{query}'")
    for i, r in enumerate(results, 1):
        print(f"  {i}. {r['product_name']:<40} [{r['category']}]")
    print()

## 3. XGBoost Learn-to-Rank

The LTR model learns to combine 12 features (BM25 score, semantic similarity, popularity, nutrition grade, query overlap, etc.) into an optimal ranking function using **LambdaMART** (pairwise ranking loss).

Key insight: the model learns that **semantic rank** is the most important feature, followed by **BM25 score** and **nutrition grade**.

In [ ]:
from src.models.ltr import LTRModel
from pathlib import Path

ltr = LTRModel(model_path=Path("../models/ltr_model.json"))

# Feature importance
importance = ltr.feature_importance()
print("Feature Importance (XGBoost LTR):\n")
for name, score in sorted(importance.items(), key=lambda x: -x[1]):
    bar = "█" * int(score * 50)
    print(f"  {name:25s} {score:.4f} {bar}")

## 4. Substitute Recommendations

When a product is out of stock, find the best alternatives — same category, similar ingredients, or healthier options.

In [ ]:
from src.recommend.substitute import SubstituteRecommender

recommender = SubstituteRecommender(catalog=catalog, embedder=embedder, embeddings=embeddings)

# Pick a popular yogurt product
yogurts = catalog[catalog["category"] == "yogurt"].sort_values("order_count", ascending=False)
product = yogurts.iloc[0]
print(f"Original: {product['product_name']} (ID: {product['product_id']})")
print(f"  Orders: {product['order_count']:,} | Reorder rate: {product['reorder_rate']:.1%}\n")

# Find similar substitutes
subs = recommender.find_substitutes(product["product_id"], top_k=5, same_category=True)
print("Top 5 Similar Substitutes:")
for i, s in enumerate(subs, 1):
    reasons = ", ".join(s.get("substitution_reasons", []))
    print(f"  {i}. {s['product_name']:<40} (sim: {s['similarity_score']:.3f})")
    if reasons:
        print(f"     → {reasons}")

In [ ]:
# Find healthier alternatives for a high-sugar product
cookies = catalog[(catalog["category"] == "cookies cakes") & catalog["sugar_100g"].notna()]
cookies = cookies.sort_values("sugar_100g", ascending=False)
product = cookies.iloc[0]
print(f"Original: {product['product_name']}")
print(f"  Sugar: {product['sugar_100g']:.1f}g/100g | Calories: {product['calories_100g']:.0f}/100g\n")

healthier = recommender.find_healthier_alternatives(product["product_id"], top_k=5)
print("Healthier Alternatives:")
for i, h in enumerate(healthier, 1):
    sugar = f"{h['sugar_100g']:.1f}g" if pd.notna(h.get("sugar_100g")) else "?"
    cal = f"{h['calories_100g']:.0f}" if pd.notna(h.get("calories_100g")) else "?"
    print(f"  {i}. {h['product_name']:<40} (sugar: {sugar}, cal: {cal})")

## 5. Evaluation — Method Comparison

Compare **Precision@5** across all search methods on 10 diverse queries.

| Method | How it works |
|--------|-------------|
| **BM25** | TF-IDF keyword matching (baseline) |
| **Hybrid (RRF)** | BM25 + semantic embeddings fused with reciprocal rank fusion |
| **XGBoost LTR** | Learned ranking over 12 features (BM25, semantic, popularity, nutrition, overlap) |

In [ ]:
import pickle

# Load pre-computed scores (saves memory vs recomputing)
with open("../data/embeddings/eval_scores.pkl", "rb") as f:
    score_data = pickle.load(f)
bm25_data, sem_data = score_data["bm25"], score_data["semantic"]

eval_queries = [
    {"query": "low sugar yogurt", "cats": ["yogurt"]},
    {"query": "organic almond milk", "cats": ["soy lactosefree"]},
    {"query": "gluten free bread", "cats": ["bread"]},
    {"query": "vegan protein bar", "cats": ["energy granola bars", "protein meal replacements"]},
    {"query": "healthy snack for kids", "cats": ["energy granola bars", "baby food formula", "fruit vegetable snacks"]},
    {"query": "Korean BBQ ingredients", "cats": ["marinades meat preparation", "asian foods", "condiments"]},
    {"query": "fresh orange juice", "cats": ["juice nectars", "refrigerated"]},
    {"query": "whole wheat pasta", "cats": ["pasta sauce", "dry pasta"]},
    {"query": "natural peanut butter", "cats": ["nuts seeds dried fruit", "spreads"]},
    {"query": "sparkling water", "cats": ["water seltzer sparkling water"]},
]

GRADE_MAP = {"a": 5, "b": 4, "c": 3, "d": 2, "e": 1}

results_table = []
for item in eval_queries:
    q, cats = item["query"], set(item["cats"])
    bs, ss = bm25_data[q], sem_data[q]

    # BM25 P@5
    bp = sum(1 for x in np.argsort(bs)[-5:][::-1] if catalog.iloc[x]["category"] in cats) / 5

    # Hybrid RRF P@5
    rrf_s = {}
    for rk, i in enumerate(np.argsort(bs)[-100:][::-1]): rrf_s[i] = rrf_s.get(i, 0) + 1/(60+rk+1)
    for rk, i in enumerate(np.argsort(ss)[-100:][::-1]): rrf_s[i] = rrf_s.get(i, 0) + 1/(60+rk+1)
    hp = sum(1 for i in sorted(rrf_s, key=lambda x: rrf_s[x], reverse=True)[:5] if catalog.iloc[i]["category"] in cats) / 5

    # LTR P@5
    br = np.argsort(np.argsort(-bs)); sr = np.argsort(np.argsort(-ss))
    qt = set(q.lower().split())
    cands = sorted(set(np.argsort(bs)[-50:][::-1].tolist()) | set(np.argsort(ss)[-50:][::-1].tolist()))
    feats = []
    for idx in cands:
        row = catalog.iloc[idx]
        rrf = 1/(60+br[idx]+1)+1/(60+sr[idx]+1)
        oc=row.get("order_count",0) or 0; rr=row.get("reorder_rate",0) or 0
        gn=GRADE_MAP.get(str(row.get("nutrition_grade","")).lower(),0)
        hn=1 if pd.notna(row.get("calories_100g")) else 0
        nt=set(str(row.get("product_name","")).lower().split())
        ct=set(str(row.get("category","")).lower().replace("_"," ").split())
        feats.append([float(bs[idx]),float(ss[idx]),rrf,float(br[idx]),float(sr[idx]),
            np.log1p(oc),rr,gn,hn,len(qt&nt)/max(len(qt),1),len(qt&ct)/max(len(qt),1),len(nt)])
    preds = ltr.model.predict(np.array(feats))
    ltr_top5 = [cands[j] for j in np.argsort(preds)[::-1][:5]]
    lp = sum(1 for i in ltr_top5 if catalog.iloc[i]["category"] in cats) / 5

    results_table.append({"Query": q, "BM25": bp, "Hybrid (RRF)": hp, "XGBoost LTR": lp})

df = pd.DataFrame(results_table)
avg = pd.DataFrame([{"Query": "AVERAGE", "BM25": df["BM25"].mean(),
                      "Hybrid (RRF)": df["Hybrid (RRF)"].mean(),
                      "XGBoost LTR": df["XGBoost LTR"].mean()}])
df = pd.concat([df, avg], ignore_index=True)
df.style.format({"BM25": "{:.2f}", "Hybrid (RRF)": "{:.2f}", "XGBoost LTR": "{:.2f}"}) \
    .apply(lambda x: ["font-weight: bold" if x.name == len(df)-1 else "" for _ in x], axis=1)

## Summary

| Method | P@5 | vs BM25 |
|--------|-----|---------|
| BM25 (baseline) | 0.820 | — |
| Hybrid (RRF) | 0.980 | +19.5% |
| **XGBoost LTR** | **1.000** | **+22.0%** |

**Key takeaways:**
- Semantic embeddings (`all-MiniLM-L6-v2`) capture query intent far better than keyword matching alone
- Reciprocal Rank Fusion effectively combines the strengths of both methods
- XGBoost LTR further improves by learning optimal feature weights (semantic rank, BM25 score, nutrition grade are top signals)
- All models run on pre-trained weights — no fine-tuning needed, keeping the project lightweight